# Smart Retail & Customer Intelligence

Exploration notebook for datasets, quick model sanity checks, and training.

Run from the project root so `app` is importable:

    jupyter lab notebooks/01_exploration.ipynb

In [ ]:
import sys
sys.path.insert(0, '..')

from app.core.logging import get_logger
logger = get_logger('notebook')

## 1. Sentiment model sanity check

In [ ]:
from app.models.sentiment_model import SentimentAnalyzer
analyzer = SentimentAnalyzer()
for text in ["I love this dress!", "Terrible quality", "It is okay I guess"]:
    print(text, '->', analyzer.analyze(text))

## 2. FAQ chatbot sanity check

In [ ]:
from app.models.chatbot_model import FAQChatbot
bot = FAQChatbot()
for q in ["How do I return an item?", "What are your hours?", "do you ship overseas"]:
    print(q, '->', bot.respond(q)[:2])

## 3. Intents overview

In [ ]:
import json
from pathlib import Path

intents = json.loads(Path('../data/intents.json').read_text())
print('intents:', len(intents['intents']))
print('tags:', [i['tag'] for i in intents['intents']])

## 4. Train the models

Each cell runs the corresponding `training/` script. Artifacts are written
to `models/artifacts/` and picked up automatically by the API on restart.

In [ ]:
import subprocess, sys

def run(*args):
    subprocess.run([sys.executable, '-m'] + list(args), check=True)

# (optional) fetch public datasets: Fashion-MNIST, reviews CSV, LFW
# run('training.download_datasets', '--all')

In [ ]:
# Sentiment: TF-IDF + Logistic Regression
# Use the real reviews CSV if available, else the built-in corpus.
from pathlib import Path
csv = Path('../data/ecommerce_reviews.csv')
args = ['training.train_sentiment']
if csv.exists():
    args += ['--csv', str(csv)]
run(*args)

In [ ]:
# Product: MobileNetV2 transfer learning on Fashion-MNIST
# --limit and --epochs keep quick runs fast; remove them for full training.
run('training.train_product', '--limit', '5000', '--epochs', '2')

> **Face recognition** is trained by enrolling 128-d encodings from a folder
> of per-person images: `python -m training.train_face data/faces --seed`.
> It requires `face_recognition` (dlib): `uv sync --group ml`.